# Imports

In [ ]:
from feature_engineering.config import PROCESSED_DATA_DIR, INTERIM_DATA_DIR

import os
import pandas
import numpy as np

# Ingest data

In [ ]:
os.listdir(INTERIM_DATA_DIR)

In [ ]:
filepath_months = INTERIM_DATA_DIR / 'meses.parquet'
filepath_nom = INTERIM_DATA_DIR / 'nomenclatura.parquet'

df_months = pandas.read_parquet(filepath_months)
df_nom = pandas.read_parquet(filepath_nom)

In [ ]:
print(df_months.shape)
df_months.head()

In [ ]:
df_months.info()

In [ ]:
df_months.describe().T

In [ ]:
print(df_nom.shape)
df_nom.head()

In [ ]:
df_nom.info()

In [ ]:
df_nom.id

# Remove unused IDs

In [ ]:
nom_id_list = df_nom.index.to_list()
df_filtered = df_months.query(
    '(origen_id in @nom_id_list) & (destino_id in @nom_id_list)'
)

print('Original row count:', df_months.shape[0])
print('Filtered row count:', df_filtered.shape[0])
print('Diff: ', df_months.shape[0] - df_filtered.shape[0])

# New features

## P1 - Regression

¿Cuántos viajes se iniciarán en una estación o zona en un periodo futuro? 

Variable objetivo: Suma de los viajes iniciados en cada estación durante cada hora.

Para construirla, la hora de inicio de cada viaje se redondea hacia abajo a la hora completa (por ejemplo, 08:47 pasa a 08:00), y se agrupan los viajes por estación de origen y hora. Las combinaciones de estación de origen y hora que no posean viaje, se registran como cero.

### Convert inicio_del_viaje to datetime

In [ ]:
df_p1 = df_filtered.copy()

In [ ]:
df_idv = df_filtered[['inicio_del_viaje']].copy()
df_idv['len_str'] = df_idv.inicio_del_viaje.str.len()

df_idv.len_str.unique()

In [ ]:
for i in [15, 16, 19]:
    display( df_idv
        .query('len_str == @i')
        .iloc[:2]
        .reset_index(drop=True)
    )

In [ ]:
dates_f1_series = pandas.to_datetime(
    df_idv.query('inicio_del_viaje.str.len() < 19').inicio_del_viaje,
    format='%d/%m/%Y %H:%M'
)

dates_f1_series

In [ ]:
dates_f2_series = pandas.to_datetime(
    df_idv.query('inicio_del_viaje.str.len() == 19').inicio_del_viaje,
    format='%Y-%m-%d %H:%M:%S'
)

dates_f2_series

In [ ]:
dates_f1_series.name = 'date_f1'
dates_f2_series.name = 'date_f2'

In [ ]:
df_idv = df_idv.join([dates_f1_series, dates_f2_series])
df_idv

In [ ]:
df_p1['inicio_del_viaje'] = df_idv.date_f1.fillna( df_idv.date_f2 )
df_p1

### Group by hour and station

In [ ]:
df_p1['hora_del_viaje'] = df_p1.inicio_del_viaje.dt.floor('h')

df_p1

In [ ]:
cols = ['viaje_id', 'hora_del_viaje', 'origen_id']

df_viajes = ( df_p1[cols]
    .groupby(['hora_del_viaje','origen_id'])
    .count()
    .reset_index()
    .rename(columns={
        'viaje_id':'cantidad_de_viajes'
    })
)

df_viajes

In [ ]:
start_date = df_viajes.hora_del_viaje.iloc[0]
end_date = df_viajes.hora_del_viaje.iloc[-1]

df_datetime = pandas.date_range(start=start_date, end=end_date, freq='h').to_frame()

In [ ]:
df_ids = pandas.DataFrame(df_p1.origen_id.sort_values().unique(), columns=['origen_id'])

In [ ]:
df_hora_estacion = df_datetime.join(df_ids, how='cross').rename(columns={0:'hora_del_viaje'})
df_hora_estacion

In [ ]:
df_p1_feature = pandas.merge(
    left=df_hora_estacion,
    right=df_viajes,
    left_on=['hora_del_viaje', 'origen_id'],
    right_on=['hora_del_viaje', 'origen_id'],
    suffixes=['', '_v'],
    how='left'
)

df_p1_feature['cantidad_de_viajes'] = (
    df_p1_feature['cantidad_de_viajes']
    .fillna(0)
    .astype('int8')
)

df_p1_feature

## P2 - Clasificación

¿Una estación estará en riesgo de quedarse sin bicicletas o sin anclajes disponibles en un periodo futuro?

Variable objetivo: Riesgo (categórica)
Clases: riesgo de vaciarse, riesgo de llenarse, sin riesgo.

Para construir la variable objetivo, el día se divide en 8 intervalos de 3 horas y cada viaje se asigna dos veces:
- Como salida de su estación de origen
- Como llegada de su estación de destino

Se suman las salidas y las llegadas de cada estación en cada intervalo, y se calcula el flujo neto por estación e intervalo utilizando: llegadas menos salidas.

Las combinaciones de estación e intervalo sin viajes se registran como cero. Por último, el flujo neto se compara con el 40 % de la capacidad de la estación: 
- Riesgo de vaciarse: flujo neto menor o igual a -40% de la capacidad.
- Riesgo de llenarse: flujo neto mayor o igual a +40% de la capacidad.
- Sin riesgo: el flujo neto está entre -40% y +40% de la capacidad, sin llegar a ninguno de los dos.

In [ ]:
df_p2 = df_filtered.copy()

### Convert str to datetime

In [ ]:
def str_to_datetime(df, col):
    df_str = df[[col]].copy()

    dates_f1_series = pandas.to_datetime(
        df_str.query(f'{col}.str.len() < 19')[col],
        format='%d/%m/%Y %H:%M'
    )
    dates_f2_series = pandas.to_datetime(
        df_str.query(f'{col}.str.len() == 19')[col],
        format='%Y-%m-%d %H:%M:%S'
    )

    dates_f1_series.name = 'date_f1'
    dates_f2_series.name = 'date_f2'

    df_str = df_str.join([dates_f1_series, dates_f2_series])

    datetime_series = df_str.date_f1.fillna( df_str.date_f2 )

    return datetime_series

In [ ]:
cols = [
    'inicio_del_viaje',
    'fin_del_viaje'
]

for col in cols:
    df_p2[col] = str_to_datetime(df_p2, col)

In [ ]:
df_p2.info()

### Group by interval

In [ ]:
cols = [
    'inicio_del_viaje',
    'fin_del_viaje'
]

for col in cols:
    df_p2[col] = df_p2[col].dt.floor('3h')

df_p2.inicio_del_viaje.unique()

In [ ]:
cols = ['viaje_id', 'fin_del_viaje', 'origen_id']

df_llegadas = ( df_p2[cols]
    .groupby(['fin_del_viaje','origen_id'])
    .count()
    .reset_index()
    .rename(columns={
        'viaje_id':'llegadas',
        'fin_del_viaje':'hora_del_viaje'
    })
)

df_llegadas

In [ ]:
cols = ['viaje_id', 'inicio_del_viaje', 'origen_id']

df_salidas = ( df_p2[cols]
    .groupby(['inicio_del_viaje','origen_id'])
    .count()
    .reset_index()
    .rename(columns={
        'viaje_id':'salidas',
        'inicio_del_viaje':'hora_del_viaje'
    })
)

df_salidas

### Fill nans with zeros

In [ ]:
start_date = df_p2.inicio_del_viaje.iloc[0]
end_date = df_p2.inicio_del_viaje.iloc[-1]

df_datetime = pandas.date_range(start=start_date, end=end_date, freq='3h').to_frame()

In [ ]:
df_ids = pandas.DataFrame(df_p2.origen_id.sort_values().unique(), columns=['origen_id'])

In [ ]:
df_hora_estacion = df_datetime.join(df_ids, how='cross').rename(columns={0:'hora_del_viaje'})

In [ ]:
df_p2_feature = pandas.merge(
    left=df_hora_estacion,
    right=df_llegadas,
    left_on=['hora_del_viaje', 'origen_id'],
    right_on=['hora_del_viaje', 'origen_id'],
    suffixes=['', '_v'],
    how='left'
)

df_p2_feature = pandas.merge(
    left=df_p2_feature,
    right=df_salidas,
    left_on=['hora_del_viaje', 'origen_id'],
    right_on=['hora_del_viaje', 'origen_id'],
    suffixes=['', '_v'],
    how='left'
)

df_p2_feature

In [ ]:
cols = ['llegadas', 'salidas']

for col in cols:
    df_p2_feature[col] = (
        df_p2_feature[col]
        .fillna(0)
        .astype('int8')
    )

df_p2_feature

### Calculate variables

In [ ]:
df_p2_feature['flujo'] = df_p2_feature['llegadas'] - df_p2_feature['salidas']

cols = ['dpcapacity']
df_p2_feature = df_p2_feature.join(
    df_nom.set_index('id')[cols],
    on='origen_id',
    how='left',
    rsuffix='_nom'
)

df_p2_feature.head()

In [ ]:
df_p2_feature['riesgo'] = df_p2_feature.flujo / (df_p2_feature.dpcapacity * 0.4)
df_p2_feature['riesgo'] = np.sign(df_p2_feature.riesgo)
df_p2_feature['riesgo'] = df_p2_feature['riesgo'].fillna(0).astype('int8')

df_p2_feature.head()

In [ ]:
df_p2_feature['riesgo'] = df_p2_feature.riesgo.replace({
    -1:'riesgo de vaciarse',
    0:'sin riesgo',
    1:'riesgo de llenarse'
})

In [ ]:
df_p2_feature.riesgo.value_counts()

## P3 - Clustering

¿Qué grupos de estaciones comparten patrones de uso y dónde se ubican en la ciudad?

No posee una variable objetivo. Lo que se busca son los patrones ocultos de uso similar entre estaciones que mostrará el clustering.

El algoritmo agrupa estaciones con perfiles parecidos basados en los movimientos de la estación. Un movimiento es cada vez que una bicicleta sale o llega a una estación. Cada viaje posee dos movimientos, con estos se calculan las variables que serán usadas en el clustering:
1. Porcentaje de movimientos, por estación y período del día (mañana, mediodía, tarde y noche)
2. Porcentaje de movimientos en días laborales, por ejemplo, 3 movimientos el lunes y 6 el sábado, sería 3/(3+6); lo que corresponde a 33.33% de movimientos en días laborales.
3. Uso diario por anclaje: es el número promedio de movimientos de una estación por día, dividido entre su capacidad (número de anclajes).
4. Balance de la mañana: indica si la estación gana o pierde bicicletas entre las 5:00 y las 12:00. Se calcula como (llegadas − salidas) ÷ (llegadas + salidas). Un valor negativo significa que salen más bicicletas de las que llegan en la mañana.
5. Balance de la tarde: el mismo cálculo anterior, con los movimientos de 16:00 a 20:00.


In [ ]:
df_p3 = df_filtered.copy()

In [ ]:
cols = [
    'inicio_del_viaje',
    'fin_del_viaje'
]

for col in cols:
    df_p3[col] = str_to_datetime(df_p3, col)

In [ ]:
df_p3.info()

In [ ]:
df_p3.head()

### 1 - Porcentaje de movimientos, por estacion y periodo del dia

In [ ]:
DAY_PERIODS = {
    "morning": (5, 12),
    "midday": (12, 16),
    "afternoon": (16, 20),
    "night": (20, 5)
}

In [ ]:
cols = ['origen_id', 'destino_id']
df_ids = pandas.concat(
    [ df_p3[[col]].rename(columns={col:'id'}) for col in cols ],
    axis=0,
    ignore_index=True
)

df_p3_feature = pandas.DataFrame(df_ids.id.sort_values().unique(), columns=['id'])

df_p3_feature

In [ ]:
col_groups = {
    'inicio': ('inicio_del_viaje', 'origen_id'),
    'fin': ('fin_del_viaje', 'destino_id'),
}

for period, hours in DAY_PERIODS.items():
    df_period = df_p3_feature.copy()
    for k, cols in col_groups.items():
        col_dt = cols[0]
        col_id = cols[1]
        start_hour = hours[0]
        end_hour = hours[1]
        df_range = df_p3.query(f'@start_hour <= {col_dt}.dt.hour < @end_hour')
        df_group = (
            df_range[['viaje_id', col_id]]
            .groupby(col_id)
            .count()
            .rename(columns={'viaje_id':k})
        )
        df_period = df_period.join(df_group, on='id', how='left')
    df_period[period] = (df_period['inicio'] + df_period['fin']).fillna(0).astype('int64')
    df_p3_feature = df_p3_feature.join(df_period[[period]], how='left')

df_p3_feature

In [ ]:
df_p3_feature['total'] = df_p3_feature.morning + df_p3_feature.midday + df_p3_feature.afternoon + df_p3_feature.night

for period in DAY_PERIODS:
    period_col = period + '_pct'
    df_p3_feature[period_col] = df_p3_feature[period] / df_p3_feature['total'] * 100

df_p3_feature.drop(columns=[ 'total', *(DAY_PERIODS.keys()) ], inplace=True)

df_p3_feature

### 2 - Porcentaje de movimientos en dias laborales

In [ ]:
df_p3

### 3 - Uso diario por anclaje

### 4 - Balance de la mañana

### 5 - Balance de la tarde